<a href="https://colab.research.google.com/github/Youssef080808/AI_Agents/blob/main/MultiAgentSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install 'smolagents[litellm]' plotly geopandas shapely kaleido -q
!pip install smolagents -U
!pip install ddgs

## Model

In [ ]:
import os
from PIL import Image
from smolagents import CodeAgent, TransformersModel, GoogleSearchTool, VisitWebpageTool

model = TransformersModel(
    model_id="Qwen/Qwen2.5-Coder-3B-Instruct",
    device_map="auto",
    max_new_tokens=2048,
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

22.82


## Agents

In [ ]:
task = """Find 5 Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe with final_answer.
Also give me some supercar factories with the same cargo plane transfer time.
Rules: Use tools given to you only; web_search returns one string, not a list, so never loop over it or process it with code: just print it and you'll see it in your next step.
In your first step, only call web_search and print the results, nothing else. Use calculate_cargo_travel_time for every travel time; never write your own distance function.
Place names don't contain coordinates, so after searching, pick 5 locations (plus 2 or 3 supercar factories), type each with its latitude and longitude as numbers by hand, like ("Glasgow, UK", (55.86, -4.25)), and call calculate_cargo_travel_time from Gotham to each one before calling final_answer."""

### Simple Agent
calculates cargo trave time

In [ ]:
from google.colab import userdata
import os
import pandas as pd
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY').strip()

search = GoogleSearchTool("serper")
search.description += " Returns ONE plain-text string, not a list. Print it and read it; don't loop over it."


def check_not_empty(final_answer, *args, **kwargs):
    if isinstance(final_answer, pd.DataFrame):
        assert not final_answer.empty, (
            "The DataFrame is empty. Print the web_search results, read them, "
            "and type the locations and their coordinates by hand."
        )
    return True

agent = CodeAgent(
    model=model,
    tools=[search, VisitWebpageTool(max_output_length=4000), calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps=6,
    max_print_outputs_length=3000,
    final_answer_checks=[check_not_empty],
)

In [ ]:
result = agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find 5 Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in  │
│ Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe with final_answer.                 │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│ Rules: Use tools given to you only; web_search returns one string, not a list, so never loop over it or process │
│ it with code: just print it and you'll see it in your next step.                                                │
│ In your first step, only call web_search and print the results, nothing else. Use calculate_cargo_travel_time   │
│ for every travel time; never write your own distance function.                                                  │
│ Place names don't contain coordinates, so after searching, pick 5 locations (plus 2 or 3 supercar factories),   │
│ type each with its latitude and longitude as numbers by hand, like ("Glasgow, UK", (55.86, -4.25)), and call    │
│ calculate_cargo_travel_time from Gotham to each one before calling final_answer.                                │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-3B-Instruct ────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  locations = web_search(query="batman filming locations")                                                         
  print(locations)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results
0. [Batman (1989) - Filming & production](https://www.imdb.com/title/tt0096895/locations/)

Filming locations ; Knebworth House, Knebworth, Hertfordshire, England, UK. (Wayne Manor; exterior) ; Acton Lane 
Power Station, Acton Lane, Acton, London, England ...

1. [The Batman | Film Locations](https://movie-locations.com/movies/b/The-Batman-2022-2.php)

The Batman filming location: Millennium Mills, Royal Victoria Dock, East London. The Batman location: Lt Gordon and
The Batman interrogate Penguin after the ...

2. [The Batman (2022) - Filming & production](https://www.imdb.com/title/tt1877830/locations/)

Filming locations ; Necropolis Cemetery, Glasgow, Scotland, UK. (Batman and Selina leaving the cemetery) · 90 ; St.
George's Hall, Liverpool, England, UK. ( ...

3. [which street is batman being filmed on? 
(glasgow)](https://www.reddit.com/r/TheBatmanFilm/comments/1w0lsxa/which_street_is_batman_being_filmed_on_glasgow/)
Date published: 2 months ago

I’m heading into glasgow today and hoping to catch a glimpse of the new movie in action! does anyone know which 
street I can see the set on? sorry if ...

4. [Batman Begins Filming Locations: Complete Guide to 
...](https://giggster.com/guide/movie-location/where-was-batman-begins-filmed)

Batman Begins was filmed in various locations across the globe, with most scenes shot in the United Kingdom, 
specifically in London, Buckinghamshire, and Essex.

5. [Batman Filming Locations | Interactive Map & Guide](https://filmingmap.com/movies/batman-filming-locations)

Where were the Batman movies filmed? Batman films have shot in Chicago, London, Liverpool, Glasgow, New York and at
Pinewood Studios, among other locations. Our ...

6. [Batman (1989)](https://movie-locations.com/movies/b/Batman-1989.php)

Batman filming location: Hatfield House, Hatfield, Hertfordshire. Batman location: 'Wayne Manor's arsenal': Marble 
Hall, Hatfield House, Hatfield ...

7. [Every Batman Filming Location in LA (Map + 
Guide)](https://www.traveltodayla.com/post/batman-movies-in-la?srsltid=AYCX9vKQuU1wsPzlKZ3PD2BOmAmTrR075uw7-5Cc7ylp
AB8vesiHMVeU)

Batman Movies and Filming Locations in LA · 01Bronson Caves · 02Warner Bros. Studio Tour · 03Greystone Mansion & 
Gardens · 04Batman: The Ride · 05Batman "Hush" ...

8. [Locations | 1966 Batman Pages](https://www.66batmania.com/guides/locations/)

The actual filming of the series was predominately undertaken at 'The Culver City Studios'. Other notable filming 
locations include; 'The Warner Brothers ...

Out: None

[Step 1: Duration 75.78 seconds| Input tokens: 2,503 | Output tokens: 1,034]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  locations = [                                                                                                    
      ("Glasgow, UK", (55.86, -4.25)),                                                                             
      ("Glasgow, Scotland", (55.86, -4.25)),                                                                       
      ("Liverpool, England", (53.40, -2.98)),                                                                      
      ("New York, USA", (40.7128, -74.0060))                                                                       
  ]                                                                                                                
                                                                                                                   
  supercar_factories = [                                                                                           
      ("Ferrari F1 Factory", (37.54, -122.40)),                                                                    
      ("Lamborghini Factory", (39.22, -122.05)),                                                                   
      ("Bentley Motors Factory", (51.47, -0.12))                                                                   
  ]                                                                                                                
                                                                                                                   
  travel_times = []                                                                                                
                                                                                                                   
  for location in locations:                                                                                       
      travel_time = calculate_cargo_travel_time((40.7128, -74.0060), location[1])                                  
      travel_times.append((location[0], travel_time))                                                              
                                                                                                                   
  for factory in supercar_factories:                                                                               
      travel_time = calculate_cargo_travel_time((40.7128, -74.0060), factory[1])                                   
      travel_times.append((factory[0], travel_time))                                                               
                                                                                                                   
  import pandas as pd                                                                                              
                                                                                                                   
  df = pd.DataFrame(travel_times, columns=["Location", "Travel Time"])                                             
  final_answer(df)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer:                  Location  Travel Time
0             Glasgow, UK         8.60
1       Glasgow, Scotland         8.60
2      Liverpool, England         8.81
3           New York, USA         1.00
4      Ferrari F1 Factory         7.07
5     Lamborghini Factory         6.93
6  Bentley Motors Factory         9.17

[Step 2: Duration 77.03 seconds| Input tokens: 5,902 | Output tokens: 1,915]

#### Plan
give agent planning capabilities

In [ ]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

## Multi-Agent System

### Web Agent

In [ ]:
web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serper"),
        VisitWebpageTool(max_ouptut_lengt=4000),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=5,
    max_print_outputs_length=3000,
)

### Manager Agent

#### Model

In [ ]:
!pip install bitsandbytes

In [ ]:

reasoning_model = TransformersModel(
    model_id="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    device_map="auto",
    max_new_tokens=2048,
)

In [ ]:
from smolagents.utils import encode_image_base64, make_image_url
from smolagents import OpenAIServerModel

def check_plot_saved(final_answer, agent_memory):
    assert os.path.exists("saved_map.png"), "Make sure to save the plot under saved_map.png!"
    return True

manager_agent = CodeAgent(
    model=reasoning_model,
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "shapely",
        "json",
        "pandas",
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_plot_saved],
    max_steps=15,
)

In [ ]:
manager_agent.visualize()

In [ ]:
manager_agent.run("""
Find 5 Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_map.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

In [ ]:
manager_agent.python_executor.state["fig"]